In modern AccFin research, valuable data is often "locked" inside company websites, regulatory documents, and other websites. While numerical datasets (CRSP, Compustat, WRDS) are well-structured and easy to query, textual and legal disclosures frequently require researchers to gather data directly from the web.

This session introduces students to web scraping in Python as a tool to collect, clean, and analyze information from online sources. Using the SEC Accounting and Auditing Enforcement Releases (AAER) archive as our central case study, you will learn how to extract structured datasets from unstructured web pages, handling the unique challenges of financial text.

The AAER example highlights both the opportunities (building novel datasets for empirical analysis) and responsibilities (respecting site terms, rate limits, and ethical research practices) that come with web scraping. By the end of the class, you will have practical skills for transforming online financial disclosures into research-ready data.

**Learning Outcomes**

By the end of this session, you will be able to:

- Understand the role of web scraping in AccFin research, and identify cases where scraping is necessary to build novel datasets.

- Apply Python libraries such as requests, BeautifulSoup, and pandas to:

- Access web pages programmatically.

- Parse HTML structures to locate relevant information.

- Store results in clean, structured formats (e.g., CSV).

- Implement scraping logic to handle pagination, nested links, and multiple fields (e.g., dates, respondents, release numbers, PDF links).

- Practice responsible scraping by setting custom User-Agents, respecting rate limits, and following ethical/legal guidelines for automated data collection.

- Clean and normalize extracted data, converting messy text into structured variables (e.g., extracting AAER numbers, release references, and “see also” links).

- Integrate scraped data into empirical workflows, linking enforcement case information to firm fundamentals, stock market data, or textual analysis pipelines.

### 0. Before you begin parsing:

Read Ethical and legal considerations:

- Respect `/robots.txt` and site terms.

- Use descriptive User-Agent headers.

- Respect rate limits (no hammering the servers).

### 1. Intro to the packages

- `requests`: fetch HTML.

- `BeautifulSoup`: parse HTML tags and extract data.

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

In [ ]:
resp = requests.get('https://www.afaanz.org/doctoral-education')
print(resp.text)

In [ ]:
soup = BeautifulSoup(resp.text, "lxml")

In [ ]:
soup.find_all("p")

In [ ]:
print(soup.find("div", class_="field-item odd").get_text(strip=True, separator="\n"))

Common tricks in `bs4`

- `.get_text(strip=True)`: removes whitespace.

- `.text`: gets raw text (may include newlines).

- `.find_all("div", class_="...")`: if multiple divisions share the same class.

- `.select("css-selector")`: powerful: supports nested lookups like div.article p:first-child.

### 2. Extact AAER cases

In [ ]:
resp = requests.get('https://www.sec.gov/enforcement-litigation/accounting-auditing-enforcement-releases', headers={"User-Agent": "AFDEN-class-demo (leye.li@unsw.edu.au)"})
soup = BeautifulSoup(resp.text, "lxml")

#### Find the table needed

In [ ]:
rows = soup.find('table', class_='usa-table views-table views-view-table cols-2').find('tbody').find_all('tr')

In [ ]:
rows

In [ ]:
rows[0]

In [ ]:
rows[0].find_all('td')[0]

In [ ]:
rows[0].find_all('td')[1].get_text(strip=True)

#### Turn the texts in the table into a Pandas Dataframe

In [ ]:
data = []
for row in rows:
    cols = row.find_all('td')
    cols = [col.get_text(strip=True) for col in cols]
    data.append(cols)

df = pd.DataFrame(data, columns=['Date', 'Respondents'])
df.head()

#### Further clean the data

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='mixed')

In [ ]:
df['defendants'] = df['Respondents'].apply(lambda x: x.split('Release No.')[0].split('; '))

In [ ]:
text = 'Timothy Daly, CPARelease No.34-103008, AAER-4568'
re.findall(r'Release No\.(.+\, AAER-[0-9]+)', text)

In [ ]:
df['Case_No'] = df['Respondents'].apply(
	lambda x: re.findall(r'Release No\.(.+\, AAER-[0-9]+)', x)[0] if re.findall(r'Release No\.(.+\, AAER-[0-9]+)', x) else None
)

#### Aggregate the previous steps into one function

In [ ]:
base_url = 'https://www.sec.gov/enforcement-litigation/accounting-auditing-enforcement-releases?page='

def Extract_AAER(page_num):
    resp = requests.get(base_url + str(page_num), headers={"User-Agent": "AFDEN-class-demo (leye.li@unsw.edu.au)"})
    soup = BeautifulSoup(resp.text, "lxml")
    rows = soup.find('table', class_='usa-table views-table views-view-table cols-2').find('tbody').find_all('tr')
    
    data = []
    for row in rows:
        cols = row.find_all('td')
        cols = [col.get_text(strip=True) for col in cols]
        data.append(cols)
    
    df = pd.DataFrame(data, columns=['Date', 'temp'])
    df['Date'] = pd.to_datetime(df['Date'], format='mixed')
    df['defendants'] = df['temp'].apply(lambda x: x.split('Release No.')[0].split('; '))
    df['Case_No'] = df['temp'].apply(
	    lambda x: re.findall(r'Release No\.(.+\, AAER-[0-9]+)', x)[0] if re.findall(r'Release No\.(.+\, AAER-[0-9]+)', x) else None
        )
    df = df.drop(columns=['temp'])
    return df

In [ ]:
Extract_AAER(12)